In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_excel("../data/raw/Online Retail.xlsx")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)

Shape: (541909, 8)

Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Data types:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object


In [7]:
# Question 1: How many rows and columns do we have?
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")


Rows: 541,909
Columns: 8


In [8]:
# Question 2: What date range does the data cover?

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f"Start: {df['InvoiceDate'].min()}")
print(f"End: {df['InvoiceDate'].max()}")
print(f"Duration: {(df['InvoiceDate'].max() - df['InvoiceDate'].min()).days} days")


Start: 2010-12-01 08:26:00
End: 2011-12-09 12:50:00
Duration: 373 days


In [9]:
# Question 3: How many unique customers are there?

print(f"Unique customers: {df['CustomerID'].nunique():,}")
print(f"Missing CustomerID: {df['CustomerID'].isnull().sum():,} rows ({df['CustomerID'].isnull().mean():.1%})")


Unique customers: 4,372
Missing CustomerID: 135,080 rows (24.9%)


In [10]:
# Question 4: How many unique products are there?

print(f"Unique StockCodes: {df['StockCode'].nunique():,}")
print(f"Unique Descriptions: {df['Description'].nunique():,}")


Unique StockCodes: 4,070
Unique Descriptions: 4,223


In [11]:
# Question 5: What countries are in the data?

print(df['Country'].value_counts().head(10))


Country
United Kingdom    495478
Germany             9495
France              8557
EIRE                8196
Spain               2533
Netherlands         2371
Belgium             2069
Switzerland         2002
Portugal            1519
Australia           1259
Name: count, dtype: int64


In [12]:
# Question 6: Are there cancelled orders?

cancelled = df[df['InvoiceNo'].astype(str).str.startswith('C')]
print(f"Cancelled orders: {len(cancelled):,} rows ({len(cancelled)/len(df):.1%})")


Cancelled orders: 9,288 rows (1.7%)


In [13]:
# Question 7: Are there negative quantities or prices?

print(f"Negative quantities: {(df['Quantity'] < 0).sum():,}")
print(f"Negative/zero prices: {(df['UnitPrice'] <= 0).sum():,}")


Negative quantities: 10,624
Negative/zero prices: 2,517


In [ ]:
# Question 8: Are there duplicates?

print(f"Duplicate rows: {df.duplicated().sum():,}")

Duplicate rows: 5,268


In [ ]:
# Question 9: What does missing data look like across all columns?

missing = df.isnull().sum()
missing_pct = df.isnull().mean() * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])


             Missing Count  Missing %
Description           1454   0.268311
CustomerID          135080  24.926694


In [ ]:
# Question 10: What are the top 10 best-selling products?

top_products = df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10)
print(top_products)


Object `products` not found.
Description
WORLD WAR 2 GLIDERS ASSTD DESIGNS     53847
JUMBO BAG RED RETROSPOT               47363
ASSORTED COLOUR BIRD ORNAMENT         36381
POPCORN HOLDER                        36334
PACK OF 72 RETROSPOT CAKE CASES       36039
WHITE HANGING HEART T-LIGHT HOLDER    35317
RABBIT NIGHT LIGHT                    30680
MINI PAINT SET VINTAGE                26437
PACK OF 12 LONDON TISSUES             26315
PACK OF 60 PINK PAISLEY CAKE CASES    24753
Name: Quantity, dtype: int64


In [ ]:
# Question 11: What does a typical transaction look like?

print(df.describe())
print("\nSample rows:")
print(df.sample(5))


Object `like` not found.
            Quantity                    InvoiceDate      UnitPrice  \
count  541909.000000                         541909  541909.000000   
mean        9.552250  2011-07-04 13:34:57.156386048       4.611114   
min    -80995.000000            2010-12-01 08:26:00  -11062.060000   
25%         1.000000            2011-03-28 11:34:00       1.250000   
50%         3.000000            2011-07-19 17:17:00       2.080000   
75%        10.000000            2011-10-19 11:27:00       4.130000   
max     80995.000000            2011-12-09 12:50:00   38970.000000   
std       218.081158                            NaN      96.759853   

          CustomerID  
count  406829.000000  
mean    15287.690570  
min     12346.000000  
25%     13953.000000  
50%     15152.000000  
75%     16791.000000  
max     18287.000000  
std      1713.600303  

Sample rows:
       InvoiceNo StockCode                          Description  Quantity  \
307853    563928     22898        CHILDRENS AP

### Key Observations

# Dataset Overview
- Shape: 541,909 rows x 8 colums
- Data Range: 2010-12-01 to 2011-12-09; 373 days

# Higghlights

UK Dominance: 
- 495,478 of 541,909 rows(91.4%) are from the United Kingdom. Other countries contributed far fewer transactions. This means that models will be heavily focused on UK.

Top Products:
- dominated by novelty/decorative items (e.g., "WORLD WAR 2 GLIDERS ASSTD DESIGNS", "WHITE HANGING HEART T-LIGHT HOLDER"), consistent with this being a wholesale gift/homeware retailer rather than general merchandise.



# Data Quality Issue Found
1. Missing Customer ID: 135, 080 rows (almost 25%) 

Action: 
- These rows cannot be used for churn - customer-level analysis, and will be dropped for Module 1.
- May still be used for demand forecasting Module 2, since that is product-level 


2. Cancelled Orders: 9, 288 rows (1.7%) 
Action: 
- InvoiveNo satrts with 'C' represent returns and cancellations which is not a real purchase
- Will be removed before both churn and demand models

3. Negative Quantities: 10,624 rows
Action:
- Mostly overlaps with cancelled orders
- Will investigate whether any negative-quantity rows are not cancellations
- Decide whether to drop or treat as data errors

4. Zero or Negative Prices: 2,517 rows
Action:
- Data errors or free promotional items
- Will drop or investigate further

5. Duplicates: 5,268 rows
Action:
- Will remove exact duplicates before aggregation



# Plan for Day 3 (Cleaning)
- Drop cancelled orders (InvoiceNo starting with 'C')
- Drop rows with missing CustomerID (for churn module)
- Remove negative quantity / zero-price rows
- Remove duplicates
- Standardize country names if needed


### Additional Observation: Extreme Outliers
- Quantity: ranges from -80,995 to 80,995 — far outside the typical range (25th-75th percentile: 1 to 10 units). Likely bulk wholesale orders or data corrections.
- UnitPrice: ranges from -11,062.06 to 38,970.00 — negative prices suggest adjustment/correction entries rather than real transactions, separate from the = zero-price rows already noted.


Action: 
- Will investigate and likely cap/filter extreme outliers during Day 3 cleaning, since they could distort both RFM features (Module 1) and demand forecasting (Module 2) if left unaddressed.